In [190]:
import torch
import gc

gc.collect()
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [191]:
import zipfile
import os

zip_path = "/content/ucsc-cse-144-spring-2026-final-project.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/data")

print(os.listdir("/content/data"))

['test', 'train', 'sample_submission.csv']


In [192]:
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset, Subset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [193]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(300, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_full = datasets.ImageFolder("/content/data/train", transform=train_transform)
val_full = datasets.ImageFolder("/content/data/train", transform=val_transform)

indices = torch.randperm(
    len(train_full),
    generator=torch.Generator().manual_seed(SEED)
).tolist()

train_size = int(0.8 * len(indices))
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(val_full, val_indices)

dataset = train_full

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Total images:", len(dataset))
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Classes:", len(dataset.classes))
print(dataset.class_to_idx)

Total images: 1079
Train images: 863
Validation images: 216
Classes: 100
{'0': 0, '1': 1, '10': 2, '11': 3, '12': 4, '13': 5, '14': 6, '15': 7, '16': 8, '17': 9, '18': 10, '19': 11, '2': 12, '20': 13, '21': 14, '22': 15, '23': 16, '24': 17, '25': 18, '26': 19, '27': 20, '28': 21, '29': 22, '3': 23, '30': 24, '31': 25, '32': 26, '33': 27, '34': 28, '35': 29, '36': 30, '37': 31, '38': 32, '39': 33, '4': 34, '40': 35, '41': 36, '42': 37, '43': 38, '44': 39, '45': 40, '46': 41, '47': 42, '48': 43, '49': 44, '5': 45, '50': 46, '51': 47, '52': 48, '53': 49, '54': 50, '55': 51, '56': 52, '57': 53, '58': 54, '59': 55, '6': 56, '60': 57, '61': 58, '62': 59, '63': 60, '64': 61, '65': 62, '66': 63, '67': 64, '68': 65, '69': 66, '7': 67, '70': 68, '71': 69, '72': 70, '73': 71, '74': 72, '75': 73, '76': 74, '77': 75, '78': 76, '79': 77, '8': 78, '80': 79, '81': 80, '82': 81, '83': 82, '84': 83, '85': 84, '86': 85, '87': 86, '88': 87, '89': 88, '9': 89, '90': 90, '91': 91, '92': 92, '93': 93, '94': 

In [194]:
idx_to_real_label = {
    idx: int(class_name)
    for class_name, idx in dataset.class_to_idx.items()
}

print(idx_to_real_label)

{0: 0, 1: 1, 2: 10, 3: 11, 4: 12, 5: 13, 6: 14, 7: 15, 8: 16, 9: 17, 10: 18, 11: 19, 12: 2, 13: 20, 14: 21, 15: 22, 16: 23, 17: 24, 18: 25, 19: 26, 20: 27, 21: 28, 22: 29, 23: 3, 24: 30, 25: 31, 26: 32, 27: 33, 28: 34, 29: 35, 30: 36, 31: 37, 32: 38, 33: 39, 34: 4, 35: 40, 36: 41, 37: 42, 38: 43, 39: 44, 40: 45, 41: 46, 42: 47, 43: 48, 44: 49, 45: 5, 46: 50, 47: 51, 48: 52, 49: 53, 50: 54, 51: 55, 52: 56, 53: 57, 54: 58, 55: 59, 56: 6, 57: 60, 58: 61, 59: 62, 60: 63, 61: 64, 62: 65, 63: 66, 64: 67, 65: 68, 66: 69, 67: 7, 68: 70, 69: 71, 70: 72, 71: 73, 72: 74, 73: 75, 74: 76, 75: 77, 76: 78, 77: 79, 78: 8, 79: 80, 80: 81, 81: 82, 82: 83, 83: 84, 84: 85, 85: 86, 86: 87, 87: 88, 88: 89, 89: 9, 90: 90, 91: 91, 92: 92, 93: 93, 94: 94, 95: 95, 96: 96, 97: 97, 98: 98, 99: 99}


In [195]:
model = models.efficientnet_b3(
    weights=models.EfficientNet_B3_Weights.DEFAULT
)

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, 512),
    nn.SiLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 100)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW([
    {"params": model.features.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3}
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6
)

best_val_acc = 0

print("EfficientNet-B3 ready")

EfficientNet-B3 ready


In [196]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/best_effb3_dropout.pth")
        print("Saved best model")

print("Best validation accuracy:", best_val_acc)

Epoch 1/30 | Train Acc: 9.62% | Val Acc: 23.61%
Saved best model
Epoch 2/30 | Train Acc: 31.75% | Val Acc: 33.80%
Saved best model
Epoch 3/30 | Train Acc: 49.36% | Val Acc: 44.91%
Saved best model
Epoch 4/30 | Train Acc: 62.69% | Val Acc: 53.70%
Saved best model
Epoch 5/30 | Train Acc: 75.32% | Val Acc: 56.48%
Saved best model
Epoch 6/30 | Train Acc: 82.39% | Val Acc: 55.56%
Epoch 7/30 | Train Acc: 88.18% | Val Acc: 59.26%
Saved best model
Epoch 8/30 | Train Acc: 92.00% | Val Acc: 60.19%
Saved best model
Epoch 9/30 | Train Acc: 93.40% | Val Acc: 62.96%
Saved best model
Epoch 10/30 | Train Acc: 93.86% | Val Acc: 63.43%
Saved best model
Epoch 11/30 | Train Acc: 97.68% | Val Acc: 63.43%
Epoch 12/30 | Train Acc: 97.80% | Val Acc: 65.28%
Saved best model
Epoch 13/30 | Train Acc: 97.91% | Val Acc: 65.74%
Saved best model
Epoch 14/30 | Train Acc: 98.49% | Val Acc: 65.74%
Epoch 15/30 | Train Acc: 98.61% | Val Acc: 68.98%
Saved best model
Epoch 16/30 | Train Acc: 99.07% | Val Acc: 66.67%
Epoch 

In [197]:
class TestDataset(Dataset):
    def __init__(self, test_dir, transform):
        self.test_dir = test_dir
        self.transform = transform

        self.image_names = sorted(
            os.listdir(test_dir),
            key=lambda x: int(x.replace(".jpg", ""))
        )

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.test_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        return image, img_name


test_dataset = TestDataset("/content/data/test", val_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Test images:", len(test_dataset))

Test images: 1036


In [198]:
model.load_state_dict(
    torch.load("/content/best_effb3_dropout.pth")
)

model.eval()

print("Best model loaded")

Best model loaded


In [199]:
fixed_predictions = []
image_ids = []

model.eval()

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        for p in preds.cpu().numpy():
            fixed_predictions.append(idx_to_real_label[p])

        image_ids.extend(names)

print("Predictions complete")

Predictions complete


In [200]:
submission = pd.DataFrame({
    "ID": image_ids,
    "Label": fixed_predictions
})

submission.to_csv("/content/submission_effb3_dropout.csv", index=False)

submission.head()

,ID,Label
0,0.jpg,65
1,1.jpg,43
2,2.jpg,38
3,3.jpg,62
4,4.jpg,42
